# Train with MLflow tracking

Goal:

- Train a YOLO model to detect license plates.
- Track parameters, metrics and artifacts in the SageMaker MLflow tracking server.
- Export the trained model to `s3://<bucket>/trains/models/`.


## Environment

`sagemaker-mlflow` is the plugin that lets `mlflow` authenticate against the
managed tracking server ARN. **Restart the kernel after this cell.**


In [ ]:
%pip install -q -U ultralytics torch torchvision onnx onnxslim mlflow sagemaker-mlflow

In [ ]:
import os
import sys
from importlib.metadata import version
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import torch
import torchvision
import ultralytics

from sagemaker.core.helper.session_helper import Session

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
RUNS = ROOT / "runs"
MODELS = ROOT / "models"

for d in (RAW, PROCESSED, RUNS, MODELS):
    d.mkdir(parents=True, exist_ok=True)

REGION = Session().boto_region_name

# written on app start, so the random bucket suffix is not hardcoded
env_file = Path.home() / ".sagemaker-yolo.env"
if "BUCKET" not in os.environ and env_file.exists():
    for line in env_file.read_text().splitlines():
        key, _, val = line.partition("=")
        os.environ.setdefault(key.strip(), val.strip())

BUCKET = os.environ["BUCKET"]

S3_RAW = f"s3://{BUCKET}/data/raw"
S3_SPLIT = f"s3://{BUCKET}/data/split"
S3_MODELS = f"s3://{BUCKET}/trains/models"

DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("python         ", sys.version.split()[0])
print("torch          ", torch.__version__)
print("torchvision    ", torchvision.__version__)
print("ultralytics    ", ultralytics.__version__)
print("mlflow         ", mlflow.__version__)
print("sagemaker-mlflow", version("sagemaker-mlflow"))
print("cuda           ", torch.cuda.is_available())
print("device         ", DEVICE)
print("region         ", REGION)
print("bucket         ", BUCKET)

Connect to the tracking server. The URI is its ARN, looked up by name so the
account id and region are not hardcoded.


In [ ]:
import boto3

from src.tracking import tracking_uri

TRACKING_URI = tracking_uri()
mlflow.set_tracking_uri(TRACKING_URI)

EXPERIMENT = "yolo-plate-detection"
experiment = mlflow.set_experiment(EXPERIMENT)

# the tracking server publishes its own UI url
server = boto3.client("sagemaker").describe_mlflow_tracking_server(
    TrackingServerName=TRACKING_URI.rsplit("/", 1)[-1]
)
UI_URL = server["TrackingServerUrl"]

print("tracking  ", TRACKING_URI)
print("status    ", server["TrackingServerStatus"])
print("experiment", EXPERIMENT, f"(id {experiment.experiment_id})")

# round-trips to the server, so a permissions problem surfaces here
print("\nexisting experiments")
for exp in mlflow.search_experiments():
    print(f"  {exp.experiment_id:>4}  {exp.name}")

# the UI opens on the empty Default experiment, so link straight to this one
print(f"\nUI: {UI_URL}/#/experiments/{experiment.experiment_id}")

## Data processing

Pull the raw data from S3, split it, push the split back.


In [ ]:
from src.data_loader import build_split, summarize, verify_split, write_data_yaml
from src.s3_sync import download, upload

LIMIT = None  # e.g. 200 for a fast smoke run
SPLIT_SEED = 0

print(download(S3_RAW, RAW))

stats = summarize(RAW)
print({k: stats[k] for k in ("pairs", "boxes_total", "boxes_per_image_max", "malformed")})

print(build_split(RAW, PROCESSED, val_fraction=0.2, limit=LIMIT, seed=SPLIT_SEED))
print(verify_split(PROCESSED))

# delete=True drops objects from a previous split
print(upload(PROCESSED, S3_SPLIT, delete=True))

names = (RAW / "classes.txt").read_text().split()
data_yaml = write_data_yaml(ROOT / "configs" / "data.yaml", PROCESSED, names)
print(data_yaml.read_text())

## Train with tracking

Ultralytics has a built-in MLflow callback driven by environment variables. It
logs hyperparameters, per-epoch metrics and the run artifacts on its own.

`MLFLOW_KEEP_RUN_ACTIVE` holds the run open after training so the dataset
context can be logged to it before it closes.


In [ ]:
import time

from ultralytics import YOLO

from src.data_loader import build_train_cfg

# ultralytics forces 0 dataloader workers on CPU, so asking for more is a no-op
train_cfg = build_train_cfg(
    device=DEVICE,
    workers=(os.cpu_count() or 2) if DEVICE != "cpu" else 0,
)
train_cfg["project"] = str(ROOT / train_cfg["project"])

cfg = {k: v for k, v in train_cfg.items() if k != "model"}
n_train = len(list((PROCESSED / "train" / "images").iterdir()))

# the ultralytics MLflow callback reads these
os.environ["MLFLOW_EXPERIMENT_NAME"] = EXPERIMENT
os.environ["MLFLOW_RUN"] = f"{DEVICE}-{n_train}img-{cfg['epochs']}ep-{cfg['imgsz']}px"
os.environ["MLFLOW_KEEP_RUN_ACTIVE"] = "true"

print(f"experiment {EXPERIMENT}")
print(f"run        {os.environ['MLFLOW_RUN']}")

model = YOLO(train_cfg["model"])

start = time.time()
results = model.train(data=str(data_yaml), **cfg)
elapsed = time.time() - start

print(f"\nelapsed: {elapsed:.0f}s ({elapsed / 60:.1f} min)")

### Log dataset context

The callback logs hyperparameters and metrics but knows nothing about the data,
so the split provenance is added to the same run before closing it.


In [ ]:
from src.tracking import log_dataset_context

save_dir = Path(results.save_dir)

logged = log_dataset_context(
    PROCESSED,
    RAW,
    **{"data.limit": str(LIMIT), "data.seed": SPLIT_SEED, "run.device": str(DEVICE)},
)
mlflow.log_metric("elapsed_seconds", elapsed)

run_id = mlflow.active_run().info.run_id
print(f"run_id {run_id}")
for key, value in logged.items():
    print(f"  {key:26} {value}")

mlflow.end_run()
print("\nrun closed")

Read the run back from the server to confirm what actually landed.


In [ ]:
fetched = mlflow.get_run(run_id)

print(f"run_id  {run_id}")
print(f"status  {fetched.info.status}")

print("\nfinal metrics")
for key in sorted(fetched.data.metrics):
    if any(m in key for m in ("mAP", "precision", "recall")):
        print(f"  {key:28} {fetched.data.metrics[key]:.4f}")

print("\ndataset params")
for key in sorted(k for k in fetched.data.params if k.startswith("data.")):
    print(f"  {key:28} {fetched.data.params[key]}")

print("\nartifacts")
for artifact in mlflow.artifacts.list_artifacts(run_id=run_id):
    print(f"  {artifact.path}")

Per-epoch history, to see whether the run converged.


In [ ]:
client = mlflow.tracking.MlflowClient()
history = client.get_metric_history(run_id, "metrics/mAP50-95B")

plt.figure(figsize=(7, 4))
plt.plot([p.step for p in history], [p.value for p in history], marker="o")
plt.xlabel("epoch")
plt.ylabel("mAP50-95")
plt.title("validation mAP50-95")
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.show()

print(f"{'epoch':>6} {'mAP50-95':>10}")
for point in history:
    print(f"{point.step:>6} {point.value:>10.4f}")

## List runs

Compare every run in the experiment, side by side.


In [ ]:
from src.tracking import compare_runs

compare_runs(EXPERIMENT)

Open the MLflow UI from the Studio launcher.

Note the UI lands on the `Default` experiment, which is empty — switch to
`yolo-plate-detection` in the sidebar, or use the direct link printed above.


## Export model

Export `best.pt` to ONNX and upload it to S3.


In [ ]:
import shutil

best = YOLO(str(save_dir / "weights" / "best.pt"))

# imgsz must match training
exported = Path(best.export(format="onnx", imgsz=train_cfg["imgsz"], opset=12, simplify=True))

n_images = sum(len(list((PROCESSED / s / "images").iterdir())) for s in ("train", "val"))

onnx_path = MODELS / (
    f"{train_cfg['name']}-{n_images}img-{train_cfg['epochs']}ep-{train_cfg['imgsz']}px.onnx"
)
shutil.move(str(exported), onnx_path)

print(onnx_path.name)
print(f"{onnx_path.stat().st_size / 1e6:.1f} MB")

Write a metadata sidecar carrying the MLflow run id, so the model file points
back at the run that produced it.


In [ ]:
import json
from datetime import datetime, timezone

sidecar = onnx_path.with_suffix(".metadata.json")
sidecar.write_text(json.dumps({
    "imgsz": train_cfg["imgsz"],
    "names": [best.names[i] for i in sorted(best.names)],
    "mlflow": {
        "run_id": run_id,
        "experiment": EXPERIMENT,
        "tracking_uri": TRACKING_URI,
    },
    "train": {k: train_cfg[k] for k in ("model", "epochs", "batch", "seed")},
    "images": n_images,
    "exported_at": datetime.now(timezone.utc).isoformat(),
}, indent=2))

print(sidecar.read_text())

Upload to S3 and attach the same files to the MLflow run.


In [ ]:
from src.s3_sync import list_objects, upload_files

dest = f"{S3_MODELS}/{onnx_path.stem}"
weights = save_dir / "weights" / "best.pt"

for uri in upload_files([onnx_path, sidecar, weights], dest):
    print(uri)

# reopen the finished run to attach the export
with mlflow.start_run(run_id=run_id):
    mlflow.log_artifact(str(onnx_path), artifact_path="export")
    mlflow.log_artifact(str(sidecar), artifact_path="export")
    mlflow.set_tag("export.s3_uri", dest)

print()
for key, meta in sorted(list_objects(dest).items()):
    print(f"  {meta['size'] / 1e6:>6.1f} MB  {key.rsplit('/', 1)[-1]}")

print(f"\nrun: {UI_URL}/#/experiments/{experiment.experiment_id}/runs/{run_id}")